# ⚽ Entrenar un detector de fútbol propio (jugadores + balón)

Entrena un modelo **YOLOv8 grande** sobre el dataset público **football-players-detection** de Roboflow
(ya etiquetado: `ball / goalkeeper / player / referee`) — **sin etiquetar nada a mano**.
El resultado es un `best.pt` que reemplaza a los modelos community actuales para **jugadores Y balón**,
apuntando a menos fragmentación de IDs y mejor detección.

> El modelo entrenado se enchufa en el pipeline con `--player-model <ruta>` y `--ball-model <ruta>`.

> ⏱️ **Entrenar es un costo de UNA vez.** Es el único momento donde conviene una GPU potente
> (L4/A100): en T4 `yolov8x@1280` puede tardar varias horas; en A100 ~1 h.


## 0. GPU


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU (activá una en Entorno de ejecución)')


## 1. Instalar dependencias


In [ ]:
!pip install -q ultralytics roboflow


## 2. Montar Drive (para guardar el modelo entrenado)


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
MODELS_DIR = '/content/drive/MyDrive/football_analytics/models'
os.makedirs(MODELS_DIR, exist_ok=True)
print('Los pesos se guardarán en:', MODELS_DIR)


## 3. API key de Roboflow (gratis)

Solo se usa para **descargar el dataset** (que es público). Sacá tu key gratis en
https://app.roboflow.com → Settings → Roboflow API → *Private API Key*.


In [ ]:
from roboflow import Roboflow
ROBOFLOW_API_KEY = ''  # <-- pegá tu key acá
assert ROBOFLOW_API_KEY, 'Falta tu Roboflow API key'
rf = Roboflow(api_key=ROBOFLOW_API_KEY)


## 4. Descargar el dataset (ya etiquetado)

Clases: `ball / goalkeeper / player / referee` — las mismas que usa el pipeline.
Si la versión da error, el mensaje lista las disponibles; ajustá `VERSION`.


In [ ]:
VERSION = 12  # ajustá si da error
project = rf.workspace('roboflow-jvuqo').project('football-players-detection-3zvbc')
dataset = project.version(VERSION).download('yolov8')
print('Dataset en:', dataset.location)

# Reescribir rutas del data.yaml a absolutas (robusto en Colab)
import yaml
yaml_path = os.path.join(dataset.location, 'data.yaml')
with open(yaml_path) as f: d = yaml.safe_load(f)
d['train'] = os.path.join(dataset.location, 'train', 'images')
d['val']   = os.path.join(dataset.location, 'valid', 'images')
d['test']  = os.path.join(dataset.location, 'test', 'images')
with open(yaml_path, 'w') as f: yaml.safe_dump(d, f)
print('Clases:', d.get('names'))


## 5. Elegir tamaño del modelo

- `m` (medium): más rápido en inferencia, buena precisión.
- `l` (large): **equilibrio recomendado** — bastante mejor que el actual, inferencia manejable.
- `x` (xlarge): **máxima precisión**, pero la inferencia (tracking de partidos) es notablemente más lenta.

Más grande = más preciso pero cada partido tarda más de procesar. Elegí según cuánto te importe la velocidad después.


In [ ]:
MODEL_SIZE = 'l'   # 'm' | 'l' | 'x'
BATCH = -1         # -1 = auto. Si da OOM (sin memoria), poné 8, 6 o 4
EPOCHS = 100


## 6. Entrenar


In [ ]:
from ultralytics import YOLO

model = YOLO(f'yolov8{MODEL_SIZE}.pt')   # transfer learning desde pesos COCO
results = model.train(
    data=os.path.join(dataset.location, 'data.yaml'),
    epochs=EPOCHS, imgsz=1280, batch=BATCH, patience=25,
    project=MODELS_DIR, name='football_detector', plots=True,
)
BEST = os.path.join(MODELS_DIR, 'football_detector', 'weights', 'best.pt')
print('\n✅ Mejor modelo guardado en Drive:', BEST)


## 7. Métricas (para saber si mejoró)


In [ ]:
metrics = model.val()
print('mAP50-95:', round(float(metrics.box.map), 3))
print('mAP50   :', round(float(metrics.box.map50), 3))

# Recall por clase (mirá especialmente el del BALÓN):
names = d['names']
items = names.items() if isinstance(names, dict) else enumerate(names)
for i, name in items:
    try:
        print(f'  {name}: recall {metrics.box.r[int(i)]:.2f} | mAP50 {metrics.box.ap50[int(i)]:.2f}')
    except Exception:
        pass


## 8. Cómo usarlo en el pipeline

En el notebook del pipeline (`colab_pipeline.ipynb`), en la celda del tracking, reemplazá:

```python
    f'--player-model football '
    f'--ball-model football '
```
por (misma ruta en los dos → detecta jugadores y balón, una sola inferencia):

```python
    f'--player-model "/content/drive/MyDrive/football_analytics/models/football_detector/weights/best.pt" '
    f'--ball-model "/content/drive/MyDrive/football_analytics/models/football_detector/weights/best.pt" '
```

El pipeline detecta las clases por nombre automáticamente, así que no hay que tocar nada más.
Después re-trackeás un clip y comparás: ¿bajan los IDs de jugador? ¿sube el balón del 78%?
